In [ ]:
import logging
import os

import pandas as pd

from napistu.utils import download_wget

from scGPT import load_scgpt, load_gene_annotations, extract_model_weights, SCGPT_DEFS
from etl_utils import (
    save_results,
    load_results,
    RESULTS_DEFS
)
from analysis_utils import (
    compute_attention_from_weights
)

logger = logging.getLogger(__name__)

In [ ]:
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
MODEL_RESULTS_PATH = os.path.join(OUTPUT_DIR, "scgpt_weights.npz")
ANNOTATIONS_PATH = os.path.join(DATA_DIR, "scgpt_gene_info.csv")

In [ ]:
# download ensembl <-> aliases mappings
if not os.path.isfile(ANNOTATIONS_PATH):
    download_wget(SCGPT_DEFS.GENE_IDENTIFIERS_URL, ANNOTATIONS_PATH)

gene_annotations = load_gene_annotations(ANNOTATIONS_PATH)

logger.info("Loading scGPT model")
model, vocab, model_metadata = load_scgpt(MODEL_PATH)

logger.info("Extracting model weights")
weights_dict = extract_model_weights(model, vocab, model_metadata)

logger.info(f"Saving weights to {OUTPUT_DIR}")
save_results(weights_dict, gene_annotations, model_metadata, OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

In [ ]:
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_11_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_11'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_11'][RESULTS_DEFS.W_K]
)